In [2]:
from pathlib import Path
import pandas as pd
import shutil
import json

# =========================
# 1. PATHS
# =========================

EXCLUDED_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/split_from_main/excluded_png"
)

CSV_PATH = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/split_from_main/excluded_png/excluded.csv"
)
# If your excluded.csv is somewhere else, drag it beside the notebook and use:
# CSV_PATH = Path("excluded.csv")

OUTPUT_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/excluded_categorized"
)

print("EXCLUDED_DIR exists:", EXCLUDED_DIR.exists())
print("CSV_PATH exists:", CSV_PATH.exists())

if not EXCLUDED_DIR.exists():
    raise FileNotFoundError(f"Excluded folder not found: {EXCLUDED_DIR}")

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

# =========================
# 2. READ CSV
# =========================

try:
    df = pd.read_csv(CSV_PATH, encoding="cp950")
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, encoding="utf-8")

print("CSV rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nReason counts:")
print(df["reason"].value_counts(dropna=False))

# =========================
# 3. CLEAN OUTPUT
# =========================

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

class_folders = [
    "nail_polish",
    "obstructed",
    "pathology",
    "review_anomaly",
    "review_unknown"
]

for folder in class_folders:
    (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)

# =========================
# 4. LABEL MAPPING
# =========================

def map_reason_to_class(reason):
    reason = str(reason).lower().strip()

    if "polish" in reason:
        return "nail_polish"

    if "occluded" in reason or "obstructed" in reason:
        return "obstructed"

    if "pathology" in reason:
        return "pathology"

    if "anomaly" in reason:
        return "review_anomaly"

    if "unknown" in reason or reason == "nan":
        return "review_unknown"

    return "review_unknown"

# =========================
# 5. BUILD LOOKUP FROM EXISTING EXCLUDED PNG FOLDER
# =========================

png_files = list(EXCLUDED_DIR.rglob("*.png"))
json_files = list(EXCLUDED_DIR.rglob("*.json"))

png_by_stem = {p.stem: p for p in png_files}
json_by_stem = {p.stem: p for p in json_files}

print("\nExisting excluded PNGs:", len(png_files))
print("Existing excluded JSONs:", len(json_files))

# =========================
# 6. COPY PNG + JSON INTO CLASS FOLDERS
# =========================

copied_pngs = 0
copied_jsons = 0
missing_pngs = []
missing_jsons = []

for _, row in df.iterrows():
    stem = str(row["stem"]).strip()
    reason = row["reason"]
    label_folder = map_reason_to_class(reason)

    src_png = png_by_stem.get(stem)

    if src_png is None:
        missing_pngs.append(stem)
        continue

    dst_folder = OUTPUT_DIR / label_folder
    dst_png = dst_folder / f"{stem}.png"

    shutil.copy2(src_png, dst_png)
    copied_pngs += 1

    src_json = json_by_stem.get(stem)

    if src_json is not None:
        dst_json = dst_folder / f"{stem}.json"

        # update imagePath to PNG name if JSON is readable
        try:
            with open(src_json, "r", encoding="utf-8") as f:
                data = json.load(f)

            if "imagePath" in data:
                data["imagePath"] = f"{stem}.png"

            with open(dst_json, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            copied_jsons += 1

        except Exception:
            shutil.copy2(src_json, dst_json)
            copied_jsons += 1
    else:
        missing_jsons.append(stem)

print("\nDone.")
print("Output folder:", OUTPUT_DIR)
print("Copied PNGs:", copied_pngs)
print("Copied JSONs:", copied_jsons)
print("Missing PNGs:", len(missing_pngs))
print("Missing JSONs:", len(missing_jsons))

if missing_pngs:
    print("\nFirst missing PNG stems:")
    for m in missing_pngs[:20]:
        print(m)

if missing_jsons:
    print("\nFirst missing JSON stems:")
    for m in missing_jsons[:20]:
        print(m)

EXCLUDED_DIR exists: True
CSV_PATH exists: True
CSV rows: 104
Columns: ['abs_path', 'relpath_from_original', 'basename', 'stem', 'size_bytes', 'mtime_iso', 'sha256_head16', 'copy_style', 'out_path', 'reason', 'notes']

Reason counts:
reason
polish                95
anomaly                3
occluded               2
unknown                2
pathology              1
occluded, anomaly      1
Name: count, dtype: int64

Existing excluded PNGs: 102
Existing excluded JSONs: 101

Done.
Output folder: /Users/williamtsai/Desktop/nail_unusable_classifier/excluded_categorized
Copied PNGs: 102
Copied JSONs: 101
Missing PNGs: 2
Missing JSONs: 1

First missing PNG stems:
20210723123050_pid2625_7wfbuG
20210630124421_pid2625_Fidi2e

First missing JSON stems:
20210727165825_pid2625_XXxKI3


In [3]:
from pathlib import Path

OUTPUT_DIR = Path("/Users/williamtsai/Desktop/nail_unusable_classifier/excluded_categorized")

for folder in sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir()]):
    pngs = list(folder.glob("*.png"))
    jsons = list(folder.glob("*.json"))
    jpgs = list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg"))

    print(folder.name)
    print("  PNG :", len(pngs))
    print("  JSON:", len(jsons))
    print("  JPG/JPEG:", len(jpgs))

nail_polish
  PNG : 95
  JSON: 95
  JPG/JPEG: 0
obstructed
  PNG : 3
  JSON: 2
  JPG/JPEG: 0
pathology
  PNG : 1
  JSON: 1
  JPG/JPEG: 0
review_anomaly
  PNG : 3
  JSON: 3
  JPG/JPEG: 0
review_unknown
  PNG : 0
  JSON: 0
  JPG/JPEG: 0
